# mlp

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        # 第一层：输入 -> 隐藏层
        self.layer1 = nn.Linear(in_features, hidden_features)
        # 激活函数：引入非线性
        self.relu = nn.ReLU()
        # 第二层：隐藏层 -> 输出
        self.layer2 = nn.Linear(hidden_features, out_features)

    def forward(self, x):
        # 定义数据流向：线性 -> 激活 -> 线性
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# 实例化一个隐藏层为 10 的 MLP
model_mlp = MLPModel(in_features=3, hidden_features=10, out_features=1)

In [ ]:
# 1. 定义模型、损失函数和优化器
model = MLPModel(3, 10, 1)
criterion = nn.MSELoss() # 官方 MSE 损失函数
optimizer = torch.optim.SGD(model.parameters(), lr=0.01) # 官方随机梯度下降优化器

# 2. 训练循环
for epoch in range(100):
    # 前向传播
    y_pred = model(X) # 等价于 model.forward(X)
    loss = criterion(y_pred, y_true)
    
    # 核心三步走（大模型代码标配）
    optimizer.zero_grad()  # 1. 梯度清零
    loss.backward()        # 2. 反向传播
    optimizer.step()       # 3. 参数更新（自动处理 no_grad 和参数减法）
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# 线性回归合订版

In [ ]:
import torch
import torch.nn as nn

# 1. 准备数据 (与之前相同)
x = torch.randn(100, 3)
true_w = torch.tensor([[2.0], [-1.0], [0.5]])
true_b = 0.5
y_true = torch.matmul(x, true_w) + true_b + 0.1 * torch.randn(100, 1)

# 2. 定义模型 (使用 nn.Module)
class LinearModel(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        # nn.Linear 自动管理了 w 和 b
        self.linear = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.linear(x)

# 实例化
model = LinearModel(in_features=3)

# 3. 定义损失函数和优化器 (这是新加入的官方工具)
criterion = nn.MSELoss()  # 均方误差损失
# SGD 优化器会自动处理 w -= lr * grad 的逻辑
optimizer = torch.optim.SGD(model.parameters(), lr=0.1) 

# 4. 训练循环
for epoch in range(100):
    # --- A. 前向传播 ---
    y_pred = model(x) # 自动调用 forward
    loss = criterion(y_pred, y_true)

    # --- B. 核心三步走 (大模型岗面试必背) ---
    optimizer.zero_grad()  # 1. 清空所有参数的梯度
    loss.backward()        # 2. 反向传播，计算梯度
    optimizer.step()       # 3. 自动更新参数 (内部处理了 no_grad)

    # --- C. 日志打印 ---
    if (epoch + 1) % 10 == 0:
        # 访问 nn.Linear 内部的参数
        curr_w = model.linear.weight.detach().squeeze().tolist()
        curr_b = model.linear.bias.item()
        print(f"Epoch [{epoch+1:>3}/100] | Loss: {loss.item():.4f} | b: {curr_b:.4f}")

# 带验证集的linear

In [ ]:
import torch
import torch.nn as nn

# 定义模型
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 1)
    def forward(self, x):
        return self.linear(x)

model = LinearModel()
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 模拟数据切分
train_X, train_Y = torch.randn(160, 3), torch.randn(160, 1)
val_X, val_Y = torch.randn(40, 3), torch.randn(40, 1)

for epoch in range(100):
    # --- 训练阶段 ---
    model.train() # 【关键】切换到训练模式
    pred = model(train_X)
    loss = criterion(pred, train_Y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # --- 验证阶段 ---
    model.eval() # 【关键】切换到评估模式
    with torch.no_grad(): # 【关键】彻底关闭梯度
        val_pred = model(val_X)
        val_loss = criterion(val_pred, val_Y)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")